[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/personal-health-agent/course-material-wip/blob/main/week-01/lab/lab.ipynb)

# Week 1 Lab: Fitbit Setup and Wearable Data Through an API

**Course:** BINF 4070 — The Future of Personal Health Assistant

**Week:** 1

## Why this lab

This lab establishes your course data path. If you choose the Fitbit path, you
set up a course Fitbit and authorize the class application for your account. If
you choose the synthetic path, you begin with the same API-shaped records
without a device. Then everyone inspects realistic Google Health API v4
DataPoint responses, traces what each nested field means, and builds a small
OpenAI summary from the selected records.

By the end, you will be able to:

- explain the device → vendor platform → authorized API boundary;
- read typed steps, heart-rate, and sleep DataPoints from nested JSON;
- distinguish choosing the Fitbit coursework path from authorizing an app;
- compare the steps, heart-rate, and sleep shapes returned for one day;
- send displayed wearable JSON to the OpenAI Responses API and check its summary.

Choose one of two coursework paths. If you accept a course Fitbit, use a
compatible personal account, pair and sync the device, and separately authorize
the class client to read the approved course scopes from your account. If you do
not want to wear or use a Fitbit, complete the same exercises with the synthetic
fixtures. Both paths cover the same concepts and grading criteria. Credentials
and live data are never a grading dependency. Never submit tokens or raw
identifiable data.

## New to Google Colab?

[Google Colab](https://colab.research.google.com/) runs a Jupyter notebook in
your browser, so you can execute Python without installing it on your computer.
A notebook contains **text cells** like this one and **code cells** with a play
button on the left.

1. Open the **Open in Colab** badge above. Sign in to a Google account if Colab
   asks, then use **File → Save a copy in Drive** so your work is saved.
2. Run cells **from top to bottom**. Click a code cell's play button or press
   **Shift+Enter**, wait for it to finish, and then continue to the next cell.
3. Do not skip the setup cell below. It installs the external Python
   libraries this notebook needs and imports everything used later.
4. Output appears directly below a code cell. A red traceback means the cell
   stopped; read its last line, fix the problem, and rerun that cell.
5. If Colab disconnects or you restart the runtime, run the notebook again from
   the setup cell before continuing.

Official help: [Colab basics](https://colab.research.google.com/notebooks/intro.ipynb)
and the [Colab FAQ](https://research.google.com/colaboratory/faq.html).

In [ ]:
# Install every external library required by this notebook inside Colab.
# Major-version bounds keep the lab reproducible without freezing old patches.
%pip install -q "requests>=2.32,<3" "openai>=2,<3"

import copy
import json
import os
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlparse

import requests
from openai import OpenAI

print("Setup complete. All required libraries are installed and imported.")

## Before the code: choose a data path and authorize access

Choose the course path that fits you:

- **Fitbit path:** accept a course Fitbit, use a compatible account, pair
  and sync the device, and separately authorize the class client for your own
  account. Google Health cannot link to a Google Workspace account, so use a
  compatible personal/non-Workspace Google Account, such as Gmail. Use that
  same exact account for Google Health, helper authorization, and live data.
  OAuth itself only authorizes the selected Google Health scopes.
- **Synthetic path:** if you do not want to wear or use a Fitbit, use the
  provided API-shaped fixtures. You complete the same parsing and summary
  exercises without a device, account, or token.

Your path choice and OAuth authorization are separate. For this class, we have
configured the Google Cloud Project, OAuth client, and account-eligibility
rules for **BINF 4070 Personal Health Data Lab**. You still decide which of the
three named read scopes to grant for your own account.

---

## Part 0 — Map the onboarding boundary

If you chose the Fitbit path, use the required account, pair the course Fitbit,
confirm that it synchronizes, and complete the API-access steps
below. If you chose the synthetic path, no device, account, or token is
required.

### Fitbit path: set up the device and account

Google supports Fitbit setup on compatible **iPhone/iPad and Android** devices.
Check the current
[Fitbit setup requirements](https://support.google.com/product-documentation/answer/14226283?hl=en).

1. Charge the course Fitbit and turn on Bluetooth on your phone.
2. Install the **Google Health app** from the Apple App Store or Google Play.
3. Open the app, choose **Sign in with Google**, and create or select a
   compatible personal Google Account, such as Gmail. Google
   Health cannot link to a Google Workspace account. Use this same exact
   personal account for the course device, helper authorization, and live data.
   If the app asks you to move an existing Fitbit account to a Google Account,
   follow the in-app migration steps.
4. In the Google Health app, open **Connections → Add device**, choose the
   Fitbit, and follow the on-screen steps. Do not pair it only from the phone's
   Bluetooth settings.
5. Confirm that the device appears in the app and shows a recent sync.

Official help: [Google Health account requirements](https://support.google.com/googlehealth/answer/14237024?hl=en&rd=1),
[set up a Fitbit device](https://support.google.com/googlehealth/answer/14236818?hl=en),
and [Fitbit setup troubleshooting](https://support.google.com/googlehealth/answer/14236619?hl=en).

> **Separate personal feature, not course setup:** Google Health can also
> [sync personal medical records](https://support.google.com/googlehealth/answer/16998660?hl=en).
> This is not needed for the course, the course OAuth client does not request
> medical-record scopes. If you use
> this feature independently, review its eligibility and privacy information
> before connecting a provider.

### Fitbit path: authorize course API access

Device pairing gets records into the account. API authorization is a separate
step that lets the class application request named record types from **your**
account.

1. Open the **BINF 4070 Personal Health Data Lab** [Google Health authorization
   helper](https://binf4070-health-oauth-helper-7pw5f3iaiq-ue.a.run.app/authorize).
2. Read the disclosure and the privacy policy and terms linked there. If you
   agree, continue with the same personal Google Account used for Google Health
   and the Fitbit, then review Google's consent screen. The app requests only
   three read-only health categories: activity and fitness for steps, health
   metrics and measurements for heart rate, and sleep for sleep sessions. The
   helper explains how each granted scope can be used in course exercises.
3. Grant any nonempty subset you are comfortable using. A live feature whose
   scope you do not grant stays unavailable; the other granted live features
   continue to work. The synthetic path covers all required work without any
   Google authorization.
4. The helper displays two values labeled `GOOGLE_HEALTH_REFRESH_TOKEN` and
   `GOOGLE_HEALTH_HELPER_TOKEN`. You will need them for the rest of the class.
   Keep that page open while you save them.
5. In Colab, click the key icon in the left sidebar to open **Secrets**. Create
   `GOOGLE_HEALTH_REFRESH_TOKEN` and `GOOGLE_HEALTH_HELPER_TOKEN`, paste the
   matching values, and enable notebook access for both.
6. Run the safe readiness check below. It reports only whether both Secret
   names are available.

The two values work as a matching credential pair. Keep both private: **never
paste them into a code cell, text cell, output, submission, or AI tool**. The
notebook exchanges the pair for a short-lived access token in memory and does
not print any credential. If either value is lost, repeat authorization. If
either may have been exposed, first revoke the class app in
[Google Account connections](https://myaccount.google.com/connections), delete
both Colab Secrets, then authorize again and store the new pair.

If authorization does not succeed, contact us through an approved private
course channel. Never post your account address, authorization output, or
credentials in the notebook or public Slack.

<details>
<summary><strong>Understand OAuth with the hotel analogy</strong></summary>

### Why OAuth needs all these pieces: the hotel analogy

Imagine that Google is a hotel and your health data are in your private room.
A personal health app cannot enter just because it says it works for you. The
hotel needs to identify the app, learn what you permit, and limit how long that
permission can be exercised. We configured the class app for this course; a
product team would configure and govern its own.

- A **Google Cloud Project** is the organization's administrative container:
  who operates the apps, which APIs are enabled, and who governs them.
- An **OAuth Client ID** is one app's registered badge. It identifies the app
  asking to act for you. In our class, this is "**BINF 4070 Personal Health Data Lab**".
- **Consent and scopes** are your instructions to the front desk: which rooms
  or actions that badge may access. Read-only activity access, for example,
  does not grant permission to change records.
- A short-lived **access token** is a temporary key card for the approved
  actions.
- A **refresh token** is a sensitive renewal pass the app can exchange for a
  new temporary key card. It avoids asking you to repeat consent every hour,
  but it must be stored carefully.

OAuth is therefore **delegated authorization**, not merely “sign in.” For this
class, we configured and govern the Cloud Project and OAuth client. If you build
a real multi-user product, your team will need to configure and govern its own Project,
client IDs, consent screen, scopes, redirect routes, credential storage, and
revocation process.

</details>

<details>
<summary><strong>Extended Reading: What changed since 2026</strong></summary>

### What changed since 2026

Google Health API v4 is the current developer surface for health and fitness
records. The legacy Fitbit Web API is scheduled to turn down in September
2026.

Google Health uses Google OAuth 2.0, and its health scopes are restricted.
Unverified applications can remain subject to a 100-user cap; wider public use
may require OAuth verification and a third-party security review. For this lab:

1. do not create an individual production app for this exercise;
2. do not paste access or refresh tokens into notebook cells;
3. use the synthetic DataPoints while learning the common parsing path.

Official references:

- [Google Health API overview](https://developers.google.com/health/about?hl=en)
- [Set up Google Cloud and OAuth](https://developers.google.com/health/setup)
- [Google Health API scopes](https://developers.google.com/health/scopes)
- [Developer and User Data Policy](https://developers.google.com/health/policies/health-api-developer-user-data-policy)
- [REST endpoints and examples](https://developers.google.com/health/endpoints)
- [Migration from the legacy Fitbit Web API](https://developers.google.com/health/migration/api-specifications?hl=en)

</details>

In [ ]:
# Safe credential check: report availability, never credential values.
def _colab_secret_is_available(name):
    if os.environ.get(name):
        return True
    try:
        from google.colab import userdata
        return bool(userdata.get(name))
    except Exception:
        # Outside Colab, a missing Secret, or notebook access not enabled.
        return False


secret_status = {
    name: _colab_secret_is_available(name)
    for name in (
        "GOOGLE_HEALTH_REFRESH_TOKEN",
        "GOOGLE_HEALTH_HELPER_TOKEN",
    )
}
for name, available in secret_status.items():
    print(f"{name}: {'available' if available else 'not available'}")

if all(secret_status.values()):
    print("Live Google Health access is ready to test; no credential was printed.")
else:
    print(
        "Live access is not ready yet. Add both Colab Secrets and enable "
        "notebook access, or continue with the equally valid synthetic path."
    )

### Checkpoint: pairing is not authorization

In one sentence, explain why pairing a device is not the same as authorizing an
application to read a health-data scope. Complete the answer cell immediately
below.

In [ ]:
# TODO: Replace the placeholder with one sentence in your own words.
pairing_vs_authorization = (
    "TODO: explain why device pairing and scoped API authorization differ."
)
print(pairing_vs_authorization)

## The teaching fixtures

The next cell creates small **API-shaped** responses. They follow the v4
DataPoint pattern:

- a list response contains dataPoints and may contain nextPageToken;
- each DataPoint may include dataSource metadata, including an optional
  application package name;
- the typed union field changes with the collection: steps, heartRate, or
  sleep;
- integer-like API values such as counts and beats per minute arrive as JSON
  strings.

They are deliberately not perfectly uniform. Real collection code must inspect
what arrived instead of assuming every date has the same fields or coverage.

In [ ]:
DATA_SOURCE = {
    "recordingMethod": "PASSIVELY_MEASURED",
    "application": {"packageName": "com.google.fitbit"},
    "device": {
        "manufacturer": "Google",
        "displayName": "Course Sample Tracker",
    },
    "platform": "FITBIT",
}


def _zulu(dt):
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def _civil(dt):
    return {
        "date": {"year": dt.year, "month": dt.month, "day": dt.day},
        "time": {"hours": dt.hour, "minutes": dt.minute},
    }


def _interval(day, minute_of_day, duration_minutes=60):
    start = datetime.fromisoformat(day).replace(tzinfo=timezone.utc)
    start += timedelta(minutes=minute_of_day)
    end = start + timedelta(minutes=duration_minutes)
    return {
        "startTime": _zulu(start),
        "startUtcOffset": "0s",
        "endTime": _zulu(end),
        "endUtcOffset": "0s",
        "civilStartTime": _civil(start),
        "civilEndTime": _civil(end),
    }


def _step_response(day, total):
    # Keep one complete Fitbit-source stream, then add a small overlapping
    # fictional import stream so the source-selection exercise is visible.
    weights = [0.08, 0.12, 0.21, 0.27, 0.19]
    counts = [round(total * w) for w in weights]
    counts.append(total - sum(counts))
    points = []
    for minute, count in zip([420, 540, 660, 780, 900, 1020], counts):
        points.append({
            "dataSource": copy.deepcopy(DATA_SOURCE),
            "steps": {
                "interval": _interval(day, minute, 120),
                "count": str(count),
            },
        })
    points.append({
        "dataSource": copy.deepcopy(DATA_SOURCE),
        "steps": {
            "interval": _interval(day, 1140, 60),
            # An on-wrist true-zero point may omit the default count.
        },
    })
    imported_source = copy.deepcopy(DATA_SOURCE)
    imported_source["application"]["packageName"] = (
        "edu.columbia.binf4070.synthetic.import"
    )
    imported_source["recordingMethod"] = "MANUAL"
    imported_source["device"]["displayName"] = "Course Sample Import"
    points.extend([
        {
            "dataSource": copy.deepcopy(imported_source),
            "steps": {
                "interval": _interval(day, 720, 60),
                "count": "250",
            },
        },
        {
            "dataSource": copy.deepcopy(imported_source),
            "steps": {
                "interval": _interval(day, 840, 60),
                "count": "0",
            },
        },
    ])
    points.sort(
        key=lambda point: point["steps"]["interval"]["startTime"],
        reverse=True,
    )  # v4 list responses are newest-first across all application sources.
    return {"dataPoints": points, "nextPageToken": ""}


def _heart_rate_response(day, values):
    points = []
    for minute, bpm in zip(
        [430, 520, 610, 700, 790, 880, 970, 1060],
        values,
    ):
        observed = datetime.fromisoformat(day).replace(tzinfo=timezone.utc)
        observed += timedelta(minutes=minute)
        points.append({
            "dataSource": copy.deepcopy(DATA_SOURCE),
            "heartRate": {
                "sampleTime": {
                    "physicalTime": _zulu(observed),
                    "utcOffset": "0s",
                    "civilTime": _civil(observed),
                },
                "metadata": {
                    "motionContext": "ACTIVE" if bpm >= 95 else "SEDENTARY",
                    "sensorLocation": "WRIST",
                },
                "beatsPerMinute": str(bpm),
            },
        })
    points.reverse()  # v4 list responses are newest-first.
    return {"dataPoints": points, "nextPageToken": ""}


def _sleep_response(day, minutes_asleep, minutes_awake):
    wake = datetime.fromisoformat(day).replace(tzinfo=timezone.utc)
    wake += timedelta(hours=7)
    start = wake - timedelta(minutes=minutes_asleep + minutes_awake)
    point = {
        "name": f"users/me/dataTypes/sleep/dataPoints/sample-{day}",
        "dataSource": copy.deepcopy(DATA_SOURCE),
        "sleep": {
            "interval": {
                "startTime": _zulu(start),
                "startUtcOffset": "0s",
                "endTime": _zulu(wake),
                "endUtcOffset": "0s",
                "civilStartTime": _civil(start),
                "civilEndTime": _civil(wake),
            },
            "type": "STAGES",
            "metadata": {
                "processed": True,
                "nap": False,
                "manuallyEdited": False,
                "stagesStatus": "SUCCEEDED",
            },
            "summary": {
                "minutesInSleepPeriod": str(minutes_asleep + minutes_awake),
                "minutesAsleep": str(minutes_asleep),
                "minutesAwake": str(minutes_awake),
            },
        },
    }
    return {"dataPoints": [point], "nextPageToken": ""}


SAMPLE_DAYS = [
    (datetime(2026, 9, 4) + timedelta(days=i)).strftime("%Y-%m-%d")
    for i in range(7)
]
step_totals = [5210, 7840, 6435, 9020, 4880, 7350, 6815]
hr_values = [
    [64, 66, 72, 98, 105, 82, 76, 69],
    [62, 65, 78, 112, 119, 90, 74, 67],
    [68, 70, 75, 88, 93, 85, 77, 71],
    [61, 64, 73, 121, 128, 96, 72, 65],
    [],  # one day has no heart-rate samples
    [65, 67, 80, 101, 108, 86, 75, 68],
    [63, 66, 74, 96, 103, 84, 73, 67],
]
sleep_values = [
    (421, 38), (388, 52), (447, 31), (365, 61),
    (432, 35), (405, 44), (414, 40),
]

SAMPLE_RESPONSES = {}
for day, steps, hrs, sleep in zip(
    SAMPLE_DAYS, step_totals, hr_values, sleep_values
):
    SAMPLE_RESPONSES[("steps", day)] = _step_response(day, steps)
    SAMPLE_RESPONSES[("heart-rate", day)] = _heart_rate_response(day, hrs)
    SAMPLE_RESPONSES[("sleep", day)] = _sleep_response(day, *sleep)

# A valid complete list response may omit the optional empty page token.
SAMPLE_RESPONSES[("sleep", SAMPLE_DAYS[1])].pop("nextPageToken")

def sample_list(data_type, day):
    key = (data_type, day)
    if key not in SAMPLE_RESPONSES:
        raise ValueError(
            f"No sample fixture for {data_type!r} on {day}. "
            f"Use {SAMPLE_DAYS[0]} through {SAMPLE_DAYS[-1]}."
        )
    return copy.deepcopy(SAMPLE_RESPONSES[key])


print(
    f"Loaded {len(SAMPLE_DAYS)} sample days: "
    f"{SAMPLE_DAYS[0]} through {SAMPLE_DAYS[-1]}."
)

### One function, two data paths

The next cell gives us one `list_health_data` function:

- `force_sample=True` returns the teaching fixture and never uses the network;
- `force_sample=False` reads your two Colab Secrets, obtains a short-lived
  access token, and requests your private Google Health data.

The function never prints a credential or puts one in a URL. Live responses
stay in your notebook runtime except when you explicitly run Part 3's OpenAI
cell. Before submission, live-data users clear the outputs that displayed raw
live JSON while keeping the generated summary and written comparison. Course
staff will read those retained materials for grading; when Fitbit data was
used, they remain derived from the student's Google Health data. If a live
Google or OpenAI call produces a traceback, clear that failed-call output too.

The API endpoint uses a kebab-case collection name such as heart-rate; the
typed JSON field uses camelCase, such as heartRate.

Live date filters use Google's civil-time fields and `YYYY-MM-DD` literals.
That makes a chosen date the wearer's local calendar day rather than a UTC
midnight window that could shift late-evening or early-morning observations.

Live list calls follow `nextPageToken` until the response is complete. Google
allows larger pages for most data types but caps sleep at 25, so this notebook
uses pageSize=1000 for steps and heart rate and pageSize=25 for sleep, then
returns one compatible wrapper containing all collected dataPoints. See the
official [v4 list method](https://developers.google.com/health/reference/rest/v4/users.dataTypes.dataPoints/list).

`granted_scopes` is a course-helper field, not a field from Google's standard
refresh response. During authorization, the helper records the exact consented
scope set in the signed helper-token pair. Its `/refresh` route validates that
pair and returns the same bound set as `granted_scopes`, even when Google omits
`scope` from a later refresh response. The notebook uses this field to gate each
live data type; it never decodes the helper token in Colab.

In [ ]:
GOOGLE_HEALTH_BASE = "https://health.googleapis.com/v4"
COURSE_HELPER_BASE_URL = "https://binf4070-health-oauth-helper-7pw5f3iaiq-ue.a.run.app"
GOOGLE_HEALTH_SCOPE_BY_DATA_TYPE = {
    "steps": "https://www.googleapis.com/auth/googlehealth.activity_and_fitness.readonly",
    "heart-rate": "https://www.googleapis.com/auth/googlehealth.health_metrics_and_measurements.readonly",
    "sleep": "https://www.googleapis.com/auth/googlehealth.sleep.readonly",
}
_HELPER_IDENTITY_SCOPES = frozenset({"openid", "email"})
_HELPER_ALLOWED_SCOPES = frozenset({
    *_HELPER_IDENTITY_SCOPES,
    *GOOGLE_HEALTH_SCOPE_BY_DATA_TYPE.values(),
})
_ACCESS_TOKEN_CACHE = {
    "value": None,
    "expires_at": 0.0,
    "granted_scopes": frozenset(),
}


class MissingGoogleHealthScope(RuntimeError):
    '''A live feature was not included in this user's granular grant.'''

    def __init__(self, data_type):
        super().__init__(f"The {data_type} live-data scope was not granted.")
        self.data_type = data_type


def _refresh_token():
    token = os.environ.get("GOOGLE_HEALTH_REFRESH_TOKEN")
    try:
        from google.colab import userdata
        token = token or userdata.get("GOOGLE_HEALTH_REFRESH_TOKEN")
    except Exception:
        # Local Jupyter has no google.colab module; Colab also raises a
        # credential-specific exception when the secret is absent or access is
        # disabled. The caller turns all of those cases into one safe message.
        pass
    return token


def _helper_token():
    token = os.environ.get("GOOGLE_HEALTH_HELPER_TOKEN")
    try:
        from google.colab import userdata
        token = token or userdata.get("GOOGLE_HEALTH_HELPER_TOKEN")
    except Exception:
        # Use the same safe missing-secret behavior as _refresh_token().
        pass
    return token


def _helper_base_url():
    url = COURSE_HELPER_BASE_URL.strip()
    parsed = urlparse(url)
    try:
        parsed.port
    except ValueError as exc:
        raise RuntimeError("The course helper URL is not a valid HTTPS origin.") from exc
    if (
        parsed.scheme != "https"
        or not parsed.netloc
        or not parsed.hostname
        or parsed.username is not None
        or parsed.password is not None
        or parsed.path not in ("", "/")
        or parsed.params
        or parsed.query
        or parsed.fragment
        or any(character.isspace() for character in url)
    ):
        raise RuntimeError(
            "The course helper URL must be an HTTPS origin with no path, "
            "credentials, query, or fragment."
        )
    return url.rstrip("/")


def _clear_access_token_cache():
    _ACCESS_TOKEN_CACHE.update(
        value=None,
        expires_at=0.0,
        granted_scopes=frozenset(),
    )


def _validated_granted_scopes(raw_scopes):
    if (
        not isinstance(raw_scopes, list)
        or not raw_scopes
        or any(not isinstance(scope, str) for scope in raw_scopes)
        or len(raw_scopes) != len(set(raw_scopes))
    ):
        raise RuntimeError(
            "The course helper returned invalid granted scopes."
        )
    granted = frozenset(raw_scopes)
    if (
        not granted.issubset(_HELPER_ALLOWED_SCOPES)
        or not _HELPER_IDENTITY_SCOPES.issubset(granted)
        or not granted.intersection(GOOGLE_HEALTH_SCOPE_BY_DATA_TYPE.values())
    ):
        raise RuntimeError(
            "The course helper returned invalid granted scopes."
        )
    return granted


def _require_live_scope(data_type):
    required_scope = GOOGLE_HEALTH_SCOPE_BY_DATA_TYPE[data_type]
    if required_scope not in _ACCESS_TOKEN_CACHE["granted_scopes"]:
        raise MissingGoogleHealthScope(data_type)


def _access_token(force_refresh=False):
    now = time.monotonic()
    if (
        not force_refresh
        and _ACCESS_TOKEN_CACHE["value"]
        and now < _ACCESS_TOKEN_CACHE["expires_at"]
    ):
        return _ACCESS_TOKEN_CACHE["value"]

    refresh_token = _refresh_token()
    if not refresh_token:
        raise RuntimeError(
            "No refresh token found. Use the synthetic path, or save your token "
            "in the GOOGLE_HEALTH_REFRESH_TOKEN Colab Secret."
        )
    helper_token = _helper_token()
    if not helper_token:
        raise RuntimeError(
            "No course helper token found. Use the synthetic path, or save the "
            "matching token in the GOOGLE_HEALTH_HELPER_TOKEN Colab Secret."
        )

    response = requests.post(
        f"{_helper_base_url()}/refresh",
        json={"refresh_token": refresh_token},
        headers={
            "Authorization": f"Bearer {helper_token}",
            "Accept": "application/json",
        },
        timeout=30,
        allow_redirects=False,
    )
    if 300 <= response.status_code < 400:
        raise RuntimeError(
            "The course helper returned an unexpected redirect. No credential "
            "was resent. Use the synthetic path and let us know."
        )
    if response.status_code >= 400:
        raise RuntimeError(
            "The course helper could not refresh access (HTTP "
            f"{response.status_code}). Do not print or submit your token. "
            "Use the synthetic path and follow the troubleshooting directions."
        )
    payload = response.json()
    if not isinstance(payload, dict):
        raise RuntimeError(
            "The course helper returned a non-object refresh response."
        )
    # granted_scopes is reconstructed by the course helper from the exact set
    # bound during initial OAuth authorization. It is a custom /refresh field,
    # not Google Health list JSON or a field Google must repeat on refresh.
    if "granted_scopes" not in payload:
        raise RuntimeError(
            "The deployed course helper is out of date: /refresh did not "
            "return the required granted_scopes field. Use the synthetic path "
            "and ask the course team to redeploy the current helper. Do not "
            "print the refresh response or any credential."
        )

    access_token = payload.get("access_token")
    expires_in = payload.get("expires_in")
    token_type = payload.get("token_type")
    granted_scopes = _validated_granted_scopes(payload["granted_scopes"])
    if not isinstance(access_token, str) or not access_token:
        raise RuntimeError("The course helper returned an empty access token.")
    if (
        str(token_type).lower() != "bearer"
        or not isinstance(expires_in, int)
        or isinstance(expires_in, bool)
        or expires_in <= 0
    ):
        raise RuntimeError(
            "The course helper returned an unsupported token response."
        )

    # Keep the access token only in runtime memory and refresh at least one
    # minute before the helper-reported expiry.
    _ACCESS_TOKEN_CACHE.update(
        value=access_token,
        expires_at=time.monotonic() + max(1, expires_in - 60),
        granted_scopes=granted_scopes,
    )
    return access_token


def _civil_day_bounds(day):
    '''Validate a local civil date before any credential or network access.'''
    try:
        day_start = datetime.strptime(day, "%Y-%m-%d")
    except (TypeError, ValueError) as exc:
        raise RuntimeError(
            "Date must be a local civil date in YYYY-MM-DD format."
        ) from exc
    if day_start.strftime("%Y-%m-%d") != day:
        raise RuntimeError(
            "Date must be a local civil date in YYYY-MM-DD format."
        )
    next_day_start = day_start + timedelta(days=1)
    return day_start, next_day_start


def list_health_data(data_type, day, force_sample=True):
    '''Return one date of typed DataPoints.

    The synthetic path is guaranteed. Private live mode is for a Fitbit-path
    student who authorized the class OAuth client for their account.
    '''
    filter_fields = {
        "steps": "steps.interval.civil_start_time",
        "heart-rate": "heart_rate.sample_time.civil_time",
        "sleep": "sleep.interval.civil_end_time",
    }
    if data_type not in filter_fields:
        raise ValueError(
            "data_type must be one of: steps, heart-rate, sleep"
        )
    day_start, next_day_start = _civil_day_bounds(day)

    if force_sample:
        return sample_list(data_type, day)

    token = _access_token()
    _require_live_scope(data_type)
    next_day = next_day_start.strftime("%Y-%m-%d")
    filter_field = filter_fields[data_type]
    base_params = {
        # Google's maximum for sleep is 25; 1000 is a conservative size for
        # the other two Week 1 data types (whose documented maximum is 10000).
        "pageSize": 25 if data_type == "sleep" else 1000,
        "filter": (
            f'{filter_field} >= "{day}" AND '
            f'{filter_field} < "{next_day}"'
        ),
    }
    endpoint = (
        f"{GOOGLE_HEALTH_BASE}/users/me/dataTypes/{data_type}/dataPoints"
    )

    def make_request(access_token, params):
        return requests.get(
            endpoint,
            headers={
                "Authorization": f"Bearer {access_token}",
                "Accept": "application/json",
            },
            params=params,
            timeout=30,
        )

    all_points = []
    page_token = None
    seen_page_tokens = set()
    refreshed_after_401 = False

    while True:
        params = dict(base_params)
        if page_token:
            params["pageToken"] = page_token

        response = make_request(token, params)
        if response.status_code == 401 and not refreshed_after_401:
            # A cached access token may have been invalidated early. Refresh
            # and retry exactly once across the complete paginated request; do
            # not automatically retry another 401, a 403, or a rate limit.
            _clear_access_token_cache()
            token = _access_token(force_refresh=True)
            _require_live_scope(data_type)
            refreshed_after_401 = True
            response = make_request(token, params)
        response.raise_for_status()
        page = response.json()
        if not isinstance(page, dict):
            raise RuntimeError(
                "Google Health returned an invalid list-response wrapper."
            )
        points = page.get("dataPoints", [])
        if not isinstance(points, list):
            raise RuntimeError(
                "Google Health returned an invalid dataPoints page."
            )
        all_points.extend(points)

        if "nextPageToken" in page:
            next_page_token = page["nextPageToken"]
            if not isinstance(next_page_token, str):
                raise RuntimeError(
                    "Google Health returned an invalid nextPageToken."
                )
        else:
            next_page_token = ""
        if not next_page_token:
            return {"dataPoints": all_points, "nextPageToken": ""}
        if next_page_token in seen_page_tokens:
            raise RuntimeError(
                "Google Health repeated a nextPageToken; pagination stopped."
            )
        seen_page_tokens.add(next_page_token)
        page_token = next_page_token


def _step_points(response):
    '''Return a validated raw step DataPoint list; [] means no observations.'''
    if not isinstance(response, dict):
        raise TypeError("A step response must be a dictionary wrapper.")
    points = response.get("dataPoints", [])
    if not isinstance(points, list):
        raise TypeError("A step response dataPoints field must be a list.")
    return points


def step_application_package(point):
    '''Return application.packageName, or None when attribution is absent.'''
    data_source = point.get("dataSource")
    if not isinstance(data_source, dict):
        return None
    application = data_source.get("application")
    if not isinstance(application, dict):
        return None
    package_name = application.get("packageName")
    if not isinstance(package_name, str) or not package_name:
        return None
    return package_name


def raw_step_total(response):
    '''Return one already-filtered raw sum, or None for no observations.'''
    points = _step_points(response)
    if not points:
        return None
    # Google may omit the default-valued count for an on-wrist true-zero point.
    return sum(int(point["steps"].get("count", "0")) for point in points)


def step_source_report(response):
    '''Return separate package totals and individual unattributed points.'''
    attributed = {}
    unattributed = []
    for index, point in enumerate(_step_points(response)):
        count = int(point["steps"].get("count", "0"))
        package_name = step_application_package(point)
        if package_name is None:
            unattributed.append({"point_index": index, "count": count})
            continue
        facts = attributed.setdefault(
            package_name,
            {"point_count": 0, "raw_step_sum": 0},
        )
        facts["point_count"] += 1
        facts["raw_step_sum"] += count
    return dict(sorted(attributed.items())), unattributed


def print_step_source_report(response):
    '''Print source-separated raw results without an all-source total.'''
    attributed, unattributed = step_source_report(response)
    if not _step_points(response):
        print("No step observations are present.")
        return
    print("Package-attributed step sources:")
    if not attributed:
        print("- none")
    for package_name, facts in attributed.items():
        print(
            f"- {package_name}: {facts['point_count']} point(s), "
            f"raw per-source sum {facts['raw_step_sum']}"
        )
    for facts in unattributed:
        print(
            f"- point index {facts['point_index']}: count {facts['count']} "
            "is unattributed and excluded because application.packageName "
            "is absent"
        )


def choose_step_application_source(
    response,
    requested_source="",
    control_name="SELECTED_STEP_APPLICATION_SOURCE",
):
    '''Validate one exact package source; auto-select only a sole source.'''
    points = _step_points(response)
    attributed, _ = step_source_report(response)
    valid_sources = list(attributed)
    if not points:
        return None
    if not valid_sources:
        raise ValueError(
            "Step DataPoints are present, but none has a nonempty "
            "dataSource.application.packageName. No source-specific total "
            "can be computed. Choose another date/path or inspect the raw "
            "points without aggregating them."
        )
    if not isinstance(requested_source, str):
        raise ValueError(f"{control_name} must be a string.")
    if requested_source:
        if requested_source not in valid_sources:
            choices = ", ".join(repr(source) for source in valid_sources)
            raise ValueError(
                f"{control_name} must exactly match one of: {choices}"
            )
        return requested_source
    if len(valid_sources) == 1:
        return valid_sources[0]
    choices = ", ".join(repr(source) for source in valid_sources)
    raise ValueError(
        f"Multiple step application sources are present. Set {control_name} "
        f"to one exact value from the source report: {choices}"
    )


def filter_steps_by_application_source(response, selected_source):
    '''Return a wrapper containing only one named application source.'''
    points = _step_points(response)
    filtered = copy.deepcopy(response)
    if not points:
        filtered["dataPoints"] = []
        return filtered
    valid_sources = {
        source
        for point in points
        if (source := step_application_package(point)) is not None
    }
    if selected_source not in valid_sources:
        choices = ", ".join(repr(source) for source in sorted(valid_sources))
        raise ValueError(
            "selected_source must exactly match a package-attributed step "
            f"source. Valid sources: {choices or 'none'}"
        )
    filtered["dataPoints"] = [
        copy.deepcopy(point)
        for point in points
        if step_application_package(point) == selected_source
    ]
    return filtered


print(
    "Google Health client ready. Use force_sample=True for the guaranteed "
    "synthetic path or force_sample=False for your authorized private data."
)

---

## Part 1 — Guided example: inspect step DataPoints

An API request has four pieces:

1. an HTTP method, here GET;
2. a resource URL;
3. an Authorization header in live mode;
4. a JSON response body.

The v4 list route is:

    GET /v4/users/me/dataTypes/steps/dataPoints

The server returns a wrapper object. Its dataPoints list contains typed
records. In the next cell, set `USE_FITBIT_DATA` to choose your authorized
Fitbit records or the synthetic fixture, and choose the local calendar date.
The synthetic path remains the default and the complete graded path. We will
inspect application sources before adding any step counts.

In [ ]:
# Choose one path for Parts 1 and 2. In Colab these appear as form controls;
# in local Jupyter, edit the values directly.
USE_FITBIT_DATA = False  # @param {type:"boolean"}
SELECTED_DATA_DATE = "2026-09-10"  # @param {type:"date"}

# Clear results from any earlier run before starting a new fetch. In Jupyter,
# a failed right-hand side does not otherwise erase an older variable value.
steps_data = None
guided_used_live = None
target_day = None
ACTIVE_USE_FITBIT_DATA = None
ACTIVE_DATA_DATE = None
PART1_FETCH_READY = False
step_points = None
first_point = None
first_interval = None
first_count = None
selected_steps_data = None
selected_step_application_source = None
selected_source_step_total = None
ACTIVE_STEP_APPLICATION_SOURCE = None
PART1_SOURCE_READY = False
recording_method = None
device_name = None
latest_end_time = None

# The same list function serves both paths; only force_sample changes.
guided_requested_live = bool(USE_FITBIT_DATA)
guided_day = SELECTED_DATA_DATE
steps_endpoint = (
    f"{GOOGLE_HEALTH_BASE}/users/me/dataTypes/steps/dataPoints"
)

steps_data = list_health_data(
    "steps", guided_day, force_sample=not guided_requested_live
)
if not steps_data.get("dataPoints", []):
    if guided_requested_live:
        raise RuntimeError(
            f"No Fitbit step DataPoints were returned for {guided_day}. "
            "Choose another synced date, or set USE_FITBIT_DATA=False and "
            "rerun Parts 1–2 to select the synthetic path explicitly."
        )
    raise RuntimeError(
        "The selected synthetic step fixture is empty; report the course "
        "material issue."
    )

guided_used_live = guided_requested_live

target_day = guided_day
ACTIVE_USE_FITBIT_DATA = guided_used_live
ACTIVE_DATA_DATE = guided_day
PART1_FETCH_READY = True

print("Would request:", steps_endpoint)
print("Data path:", "Fitbit" if guided_used_live else "synthetic")
print("Selected date:", target_day)
print("Top-level keys:", list(steps_data))
print(json.dumps(steps_data, indent=2)[:1800])

### 1.1 Separate application sources before counting

One list response can contain step points written by more than one application.
Adding every point together can therefore double-count overlapping records.
The next cell prints each distinct `dataSource.application.packageName` with
its own raw sum. It never prints one all-source sum. Because source metadata is
optional, it also prints every unattributed point separately and excludes it
from package totals rather than treating unrelated missing sources as one
source.

These are raw per-package results for learning the list schema, not an official
reconciled daily total. Production systems should use Google Health's
`dataPoints:reconcile` and `dataPoints:dailyRollUp` workflows when they need a
reconciled daily value.

In [ ]:
# Inspect package sources and their separate raw results before choosing one.
print_step_source_report(steps_data)

### 1.2 Choose one source, then follow its JSON path

The fixture defaults to `com.google.fitbit`. If the report shows one package
source, you may also leave the control blank and the notebook selects it
automatically. If it shows more than one, choose one exact package name in
`SELECTED_STEP_APPLICATION_SOURCE` and rerun this cell. An invalid name stops
with the valid choices. The code filters first and only then reads or adds
counts from the selected points.

Python treats JSON objects as dictionaries and JSON arrays as lists. Start at
the filtered wrapper, select the first list element, then select its typed
steps object. When present, `steps.count` is encoded as a string. Google may
omit its default value for an on-wrist true-zero point, so this notebook uses
zero for that returned point. An empty `dataPoints` list is no observation.

In [ ]:
# Leave blank to auto-select only when the report has exactly one package.
SELECTED_STEP_APPLICATION_SOURCE = "com.google.fitbit"  # @param {type:"string"}

# Clear earlier derived values before validating a new source choice.
selected_steps_data = None
selected_step_application_source = None
selected_source_step_total = None
step_points = None
first_point = None
first_interval = None
first_count = None
ACTIVE_STEP_APPLICATION_SOURCE = None
PART1_SOURCE_READY = False

selected_step_application_source = choose_step_application_source(
    steps_data,
    SELECTED_STEP_APPLICATION_SOURCE,
    "SELECTED_STEP_APPLICATION_SOURCE",
)
# Store the resolved value so later cells can identify the exact source used.
SELECTED_STEP_APPLICATION_SOURCE = selected_step_application_source
selected_steps_data = filter_steps_by_application_source(
    steps_data, selected_step_application_source
)
step_points = selected_steps_data["dataPoints"]

# Follow the typed steps field on the first observation to its time interval.
first_point = step_points[0]
first_interval = first_point["steps"]["interval"]

# Google represents integer-like counts as strings and may omit a true zero.
first_count = int(first_point["steps"].get("count", "0"))

# Add only the points from the one selected package source.
selected_source_step_total = raw_step_total(selected_steps_data)
ACTIVE_STEP_APPLICATION_SOURCE = selected_step_application_source
PART1_SOURCE_READY = True

print("Selected application source:", selected_step_application_source)
print("Number of selected intervals:", len(step_points))
print("First interval starts:", first_interval["startTime"])
print("First interval count:", first_count)
print("Raw sum for the selected source:", selected_source_step_total)

In [ ]:
# Verify the fully worked example.
assert isinstance(steps_data, dict)
assert isinstance(step_points, list) and step_points
assert all("steps" in point for point in step_points)
assert all(
    step_application_package(point) == selected_step_application_source
    for point in step_points
)
raw_source_totals, raw_unattributed_steps = step_source_report(steps_data)
assert raw_unattributed_steps == []
raw_step_start_times = [
    point["steps"]["interval"]["startTime"]
    for point in steps_data["dataPoints"]
]
assert raw_step_start_times == sorted(raw_step_start_times, reverse=True)
assert {
    point["dataSource"]["recordingMethod"]
    for point in steps_data["dataPoints"]
} == {"PASSIVELY_MEASURED", "MANUAL"}
assert set(raw_source_totals) == {
    "com.google.fitbit",
    "edu.columbia.binf4070.synthetic.import",
}
assert raw_source_totals["com.google.fitbit"]["raw_step_sum"] == (
    selected_source_step_total
)
assert raw_source_totals[
    "edu.columbia.binf4070.synthetic.import"
]["raw_step_sum"] == 250
assert selected_source_step_total == sum(
    int(point["steps"].get("count", "0")) for point in step_points
)
assert raw_step_total({"dataPoints": []}) is None
assert raw_step_total({
    "dataPoints": [{"steps": {"count": "0"}}]
}) == 0
assert raw_step_total({"dataPoints": [{"steps": {}}]}) == 0
print("Part 1 checks passed.")

### 1.3 Scaffolded JSON navigation

Extract three facts without copying the printed JSON by hand:

- the recording method of the first point, or `unavailable` when omitted;
- the device display name, or `unavailable` when omitted;
- the latest interval end time.

Use dictionary keys and list indices so the code still works if the values
change. Google Health can omit the data-source metadata fields, and list
responses are newest-first; compute the latest end explicitly instead of
assuming it is at index `-1`.

In [ ]:
# TODO: Complete the helper using safe .get(...) calls and max(...).
def describe_steps_wrapper(response):
    '''Return recording method, device name, and latest end time.'''
    raise NotImplementedError("Complete describe_steps_wrapper")

recording_method, device_name, latest_end_time = describe_steps_wrapper(
    selected_steps_data
)
print(recording_method, device_name, latest_end_time)

In [ ]:
# Verify your JSON navigation.
assert isinstance(recording_method, str) and recording_method
assert isinstance(device_name, str) and device_name
assert latest_end_time.endswith("Z")

# Optional provenance fields may be absent in a valid point.
minimal_steps = copy.deepcopy(selected_steps_data)
minimal_steps["dataPoints"][0]["dataSource"] = {}
minimal_facts = describe_steps_wrapper(minimal_steps)
assert minimal_facts[0] == "unavailable"
assert minimal_facts[1] == "unavailable"
assert minimal_facts[2] == max(
    point["steps"]["interval"]["endTime"]
    for point in minimal_steps["dataPoints"]
)
print("Part 1.3 checks passed.")

---

## Part 2 — Explore heart-rate and sleep shapes [TODOs for you]

The endpoint pattern stays stable, but the typed field changes. Heart-rate is
a set of sample measurements; sleep is a session with a summary. Your code
must follow the schema of the requested collection rather than search blindly
for a generic value field. Part 2 uses the path and date that Part 1 actually
loaded. A selected Fitbit path never changes to synthetic data automatically.
The source-selection rule above applies to step counting; Part 2 does not add
heart-rate or sleep points together.

In [ ]:
# Reuse the path and date that Part 1 actually loaded.
if not PART1_FETCH_READY or not PART1_SOURCE_READY:
    raise RuntimeError(
        "Part 1 did not complete its current fetch and step-source choice. "
        "Rerun Part 1 before Part 2."
    )

part2_day = ACTIVE_DATA_DATE
part2_force_sample = not ACTIVE_USE_FITBIT_DATA
heart_rate_data = None
sleep_data = None
heart_rate_points = None
sleep_points = None
heart_rates = None
sleep_summary = None
PART2_FETCH_READY = False
_part2_candidate = None


def fetch_part2_type(data_type):
    '''Fetch one selected type without changing the student's data path.'''
    response = list_health_data(
        data_type, part2_day, force_sample=part2_force_sample
    )

    if not response.get("dataPoints", []):
        path_label = "synthetic" if part2_force_sample else "Fitbit"
        print(
            f"No {data_type} DataPoints are present for {part2_day} on the "
            f"{path_label} path. This is no observation, not a measured zero."
        )
    return response


_part2_candidate = {
    data_type: fetch_part2_type(data_type)
    for data_type in ("heart-rate", "sleep")
}
heart_rate_data = _part2_candidate["heart-rate"]
sleep_data = _part2_candidate["sleep"]

# Show the typed key only when the selected response contains a point.
heart_rate_points = heart_rate_data.get("dataPoints", [])
sleep_points = sleep_data.get("dataPoints", [])
PART2_FETCH_READY = True
print(
    "Heart-rate point keys:",
    list(heart_rate_points[0]) if heart_rate_points else "unavailable",
)
print(
    "Sleep point keys:",
    list(sleep_points[0]) if sleep_points else "unavailable",
)

### 2.1 Extract heart-rate values

Complete the function. It must return integers and must also work when the
response contains zero dataPoints.

In [ ]:
def extract_heart_rates(response):
    '''Return all beats-per-minute values as integers.'''
    # TODO: Iterate over response["dataPoints"] and follow heartRate.
    # Hint: int(point["heartRate"]["beatsPerMinute"])
    # Expected output: a list of integers, or [] when this type is unavailable.
    raise NotImplementedError("Implement extract_heart_rates")


heart_rates = extract_heart_rates(heart_rate_data)
print(heart_rates)

In [ ]:
# Verify heart-rate extraction, including an empty valid response.
assert all(isinstance(value, int) for value in heart_rates)
assert extract_heart_rates({"dataPoints": []}) == []
if heart_rates:
    print(f"Extracted {len(heart_rates)} heart-rate samples.")
else:
    print("Heart-rate values are unavailable for the selected path and date.")
print("Part 2.1 checks passed.")

### 2.2 Summarize a sleep session

Return None when no session is present. Otherwise return minutes asleep,
minutes awake, and whether processing is complete.

In [ ]:
def summarize_sleep(response):
    '''Return a compact sleep dictionary, or None for no session.'''
    # TODO: Inspect each DataPoint for a sleep.summary with the required fields.
    # Metadata is optional. Convert minute strings to integers.
    # Expected keys: minutes_asleep, minutes_awake, processed.
    raise NotImplementedError("Implement summarize_sleep")


sleep_summary = summarize_sleep(sleep_data)
print(sleep_summary)

In [ ]:
# Verify sleep extraction and the no-session branch.
if sleep_summary is not None:
    assert sleep_summary["minutes_asleep"] > 0
    assert sleep_summary["minutes_awake"] >= 0
    assert isinstance(sleep_summary["processed"], bool)
    print("Extracted the selected sleep session.")
else:
    print("Sleep values are unavailable for the selected path and date.")
assert summarize_sleep({"dataPoints": []}) is None
assert summarize_sleep({
    "dataPoints": [
        {"sleep": {}},
        {"sleep": {
            "summary": {"minutesAsleep": "0", "minutesAwake": "0"}
        }},
    ]
}) == {"minutes_asleep": 0, "minutes_awake": 0, "processed": False}
print("Part 2.2 checks passed.")

---

## Part 3 — Send selected-day wearable data to OpenAI

This is a separate exercise from Google OAuth. Choose synthetic data or your
Fitbit data, choose one local calendar date, and run the next cell. It fetches
the raw v4 list responses for steps, heart rate, and sleep, then displays the
exact JSON before any OpenAI request. Before display, it prints the available
step application packages and filters the steps wrapper to one selected
package. Heart-rate and sleep remain unchanged because this exercise does not
add their points together.

The selected path is literal: a Fitbit fetch failure stops this cell before it
displays data or enables an OpenAI request. To use fixtures instead, set the
Fitbit control to `False` and rerun the cell explicitly.

If you select Fitbit data and explicitly run the OpenAI summary cell, the
displayed raw responses are sent to OpenAI as the input for the summary
feature. Those responses may include precise interval or sample timestamps,
DataPoint names, and device or data-source metadata. OAuth and the course helper
send nothing to OpenAI, and Google/OAuth credentials are never included. The
Responses requests below use `store=False`. OpenAI access is optional; the
notebook remains runnable without a key. When a key is present, an API failure
surfaces the SDK traceback instead of being hidden by a generic error message.

Official references: [Python quickstart](https://developers.openai.com/api/docs/quickstart),
[text generation with the Responses API](https://developers.openai.com/api/docs/guides/text),
and [`gpt-5.6-luna`](https://developers.openai.com/api/docs/models/gpt-5.6-luna).

### 3.1 Select and display one day

In [ ]:
# Part 3 has its own explicit data choice and date.
USE_FITBIT_FOR_OPENAI = False  # @param {type:"boolean"}
OPENAI_DATA_DATE = "2026-09-10"  # @param {type:"date"}
# Leave blank to auto-select only when steps have exactly one package source.
OPENAI_STEP_APPLICATION_SOURCE = "com.google.fitbit"  # @param {type:"string"}

# Invalidate any earlier displayed payload before a new fetch begins.
openai_selected_data = None
openai_displayed_json = None
OPENAI_DISPLAY_SELECTION = None
OPENAI_DISPLAY_READY = False
OPENAI_SELECTED_STEP_SOURCE = None
summary_response = None
generated_summary = None
_openai_raw_candidate_data = None
_openai_candidate_data = None
_openai_candidate_json = None

_openai_raw_candidate_data = {
    data_type: list_health_data(
        data_type,
        OPENAI_DATA_DATE,
        force_sample=not USE_FITBIT_FOR_OPENAI,
    )
    for data_type in ("steps", "heart-rate", "sleep")
}
print_step_source_report(_openai_raw_candidate_data["steps"])
OPENAI_SELECTED_STEP_SOURCE = choose_step_application_source(
    _openai_raw_candidate_data["steps"],
    OPENAI_STEP_APPLICATION_SOURCE,
    "OPENAI_STEP_APPLICATION_SOURCE",
)
# Store the resolved value so changing or rerunning the source control cannot
# reuse a payload displayed for a different package.
OPENAI_STEP_APPLICATION_SOURCE = OPENAI_SELECTED_STEP_SOURCE or ""
_openai_candidate_data = dict(_openai_raw_candidate_data)
_openai_candidate_data["steps"] = filter_steps_by_application_source(
    _openai_raw_candidate_data["steps"],
    OPENAI_SELECTED_STEP_SOURCE,
)
_openai_candidate_json = json.dumps(_openai_candidate_data, indent=2)

print(
    "Part 3 data path:",
    "Fitbit" if USE_FITBIT_FOR_OPENAI else "synthetic",
)
print("Part 3 date:", OPENAI_DATA_DATE)
print(
    "Selected step application source:",
    OPENAI_SELECTED_STEP_SOURCE or "no step observations",
)
print("Displayed raw JSON:")
print(_openai_candidate_json)

openai_selected_data = _openai_candidate_data
openai_displayed_json = _openai_candidate_json
OPENAI_DISPLAY_SELECTION = (
    bool(USE_FITBIT_FOR_OPENAI),
    OPENAI_DATA_DATE,
    OPENAI_SELECTED_STEP_SOURCE,
)
OPENAI_DISPLAY_READY = True

### 3.2 Check the key and make a first call

In Colab, open **Secrets** with the key icon, create `OPENAI_API_KEY`, paste the
OpenAI API key issued individually to you, and enable notebook access. Never
paste the key into a notebook cell or share it with anyone. The next cell
checks only whether the key is available. When it is, the
cell sends a small `Hello world!` request so you can confirm the API works.

In [ ]:
def read_private_secret(name):
    '''Read a Colab Secret or local environment variable without printing it.'''
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


openai_api_key = read_private_secret("OPENAI_API_KEY")
hello_response = None
print(
    "OPENAI_API_KEY:",
    "available" if openai_api_key else "not available",
)

if openai_api_key:
    openai_client = OpenAI(
        api_key=openai_api_key,
        timeout=30.0,
        max_retries=0,
    )
    hello_response = openai_client.responses.create(
        model="gpt-5.6-luna",
        input="Hello world!",
        reasoning={"effort": "none"},
        max_output_tokens=80,
        store=False,
    )
    print("OpenAI test response:", hello_response.output_text)
else:
    openai_client = None
    print("OpenAI test skipped because no key is available.")

#### Separate the instructions from the displayed data

The request below places the task, date, source-counting rule, and safety rules
in `instructions`. It passes the exact JSON printed in Part 3.1 through `input`.
The steps wrapper in that JSON contains only the selected package source. This
small separation makes the rules distinct from the data being summarized. If
your key is available, the cell prints `response.output_text`; otherwise it
prints a skip message.

In [ ]:
def build_summary_instructions(day, step_application_source):
    if step_application_source is None:
        step_source_rule = (
            "The steps wrapper is empty. Report no step observations; do not "
            "turn missing observations into a measured zero."
        )
    else:
        step_source_rule = (
            "The steps wrapper was filtered to one application.packageName "
            "before display. Use only those points, keep the result labeled "
            "as a raw per-source sum, and do not combine application sources."
        )
    return f'''Task:
Summarize the wearable data for {day} in clear language.

Requirements:
- Report what the data shows for steps, heart rate, and sleep.
- {step_source_rule}
- Say when a data type is empty or missing.
- Do not repeat DataPoint names, exact timestamps, or device/data-source metadata.
- Do not diagnose a condition or give medical advice.
'''.strip()


summary_instructions = build_summary_instructions(
    OPENAI_DATA_DATE, globals().get("OPENAI_SELECTED_STEP_SOURCE")
)
summary_response = None
generated_summary = None

if openai_client is not None:
    current_selection = (
        bool(USE_FITBIT_FOR_OPENAI),
        OPENAI_DATA_DATE,
        OPENAI_SELECTED_STEP_SOURCE,
    )
    if (
        not globals().get("OPENAI_DISPLAY_READY", False)
        or not isinstance(globals().get("openai_displayed_json"), str)
        or not openai_displayed_json
        or globals().get("OPENAI_DISPLAY_SELECTION") != current_selection
        or globals().get("OPENAI_STEP_APPLICATION_SOURCE")
        != (globals().get("OPENAI_SELECTED_STEP_SOURCE") or "")
    ):
        raise RuntimeError(
            "Part 3.1 did not finish for the current Fitbit/date/source "
            "selection. Rerun Part 3.1 before sending anything to OpenAI."
        )
    summary_response = openai_client.responses.create(
        model="gpt-5.6-luna",
        instructions=summary_instructions,
        input=openai_displayed_json,
        reasoning={"effort": "none"},
        max_output_tokens=500,
        store=False,
    )
    generated_summary = summary_response.output_text
    print(generated_summary)
else:
    generated_summary = None
    print("Summary call skipped because OPENAI_API_KEY is not available.")

### 3.3 Compare the summary with the displayed data

Write one short paragraph below. Compare the generated summary with the raw
JSON printed in Part 3.1, and note anything that is missing, wrong, or
surprising. If it repeats a DataPoint name, exact timestamp, or device/data-source
detail, remove that detail from the summary you retain before CourseWorks. If
the call was skipped, say what you would check when you run it. Course staff
will read the retained summary and this comparison for grading. If you used
Fitbit input, both are derived from your Google Health data even after you
clear the raw JSON output.

In [ ]:
# TODO: Replace the placeholder with one comparison paragraph.
summary_comparison = '''
TODO: compare the generated summary with the displayed steps, heart-rate, and
sleep JSON; note anything missing, wrong, or surprising.
'''.strip()
print(summary_comparison)

---

## Take-home — Build a selected-day summary

Complete one straightforward workflow:

- **Fitbit path:** choose one day of your authorized data and retrieve its raw
  steps, heart-rate, and sleep list responses.
- **Synthetic path:** load the three provided weekly fixtures below. They are
  fictional but API-shaped v4 list responses covering 2026-09-04 through
  2026-09-10.

The fictional fixtures represent one plausible week for a 24-year-old male
college student; that design context is not stored as extra JSON fields. The
week is an example, not a normative baseline. Inspect its day-to-day activity,
heart-rate, and sleep variation, and consider what the sparse samples do not
show.

Then:

1. choose one date;
2. inspect the step application packages, choose one exact package when more
   than one is present, and filter the steps wrapper before counting or display;
3. print the source-filtered steps plus the selected heart-rate and sleep JSON
   as intermediate output;
4. build your own instructions or reuse `build_summary_instructions`;
5. call OpenAI when your key is available and print the final summary;
6. complete the two-question summary and reflection at the end;
7. review the retained summary and remove any repeated DataPoint name, exact
   timestamp, or device/data-source detail;
8. if you used live Fitbit data, clear every raw-live-JSON output before
   download, and clear any failed live-provider traceback output, while keeping
   the reviewed summary and written reflection;
9. remember that course staff will read the retained summary, comparison, and
   reflection for grading; with Fitbit input, these remain derived from your
   Google Health data;
10. submit only this notebook to CourseWorks.

Your code may reuse functions from the in-class work. Keep the implementation
short and visible; do not include credentials in the prompt or submission.

In [ ]:
# Load the three weekly wrappers locally or from the course repository in Colab.
WEEKLY_DATA_BASE = (
    "https://raw.githubusercontent.com/personal-health-agent/"
    "course-material-wip/main/week-01/lab/data"
)


def load_weekly_fixture(filename):
    local_path = Path("data") / filename
    if local_path.exists():
        return json.loads(local_path.read_text(encoding="utf-8"))
    response = requests.get(f"{WEEKLY_DATA_BASE}/{filename}", timeout=30)
    response.raise_for_status()
    return response.json()


weekly_steps = load_weekly_fixture("steps.json")
weekly_heart_rate = load_weekly_fixture("heart-rate.json")
weekly_sleep = load_weekly_fixture("sleep.json")
print(
    "Loaded weekly points:",
    len(weekly_steps.get("dataPoints", [])),
    len(weekly_heart_rate.get("dataPoints", [])),
    len(weekly_sleep.get("dataPoints", [])),
)

In [ ]:
# TODO: Complete the take-home workflow described above.
# Choose Fitbit or synthetic data and one date, print all three selected raw
# wrappers, choose and filter one exact step application source, build/reuse
# instructions, call OpenAI when the key is available, and print the summary.
takehome_use_fitbit = False
takehome_date = "2026-09-10"
takehome_step_application_source = "com.google.fitbit"

raise NotImplementedError("Complete the take-home workflow")

## Troubleshooting

| Problem | What to do |
|---|---|
| ModuleNotFoundError for requests | Rerun the setup cell, then restart the runtime only if Colab asks. |
| KeyError while navigating JSON | Print the top-level keys, then inspect one DataPoint. Do not assume every collection has the same typed field. |
| Empty dataPoints | Treat it as a valid no-observation response unless the HTTP request itself failed. |
| More than one step application source | Read the source report, choose one exact `application.packageName` in the next source control, and rerun. Never add the package totals together. |
| A step source choice is rejected | Copy one exact package name from the printed valid-source list. Points without `application.packageName` are shown separately and excluded because they cannot be safely assigned to one app source. |
| No refresh token found | In Colab's Secrets panel, add `GOOGLE_HEALTH_REFRESH_TOKEN`, enable notebook access, and rerun the helper cell. Never paste the value into a notebook cell. |
| No course helper token found | Add the matching `GOOGLE_HEALTH_HELPER_TOKEN`, enable notebook access, and rerun the helper cell. Never paste the value into a notebook cell. |
| Helper says the deployed version is out of date | The `/refresh` response is missing its required course-specific `granted_scopes` field. Use the synthetic path and ask us to redeploy the current helper. Never print the response or a credential. |
| OpenAI key is unavailable | Add `OPENAI_API_KEY` in Colab Secrets and enable notebook access, or leave the optional API calls skipped. Never paste the key into a cell. |
| A live Google or OpenAI call raises a traceback | Read the traceback in Colab and correct the named input, authorization, network, or provider problem. The notebook does not change data paths or replace failed responses. If Fitbit data was selected, clear the failed-call output before submission and never post it publicly. |
| Helper returns 400 or 401 | The two credentials may be mismatched, expired, invalid, or revoked. Stop retrying, use the synthetic path, and repeat authorization to replace both values. |
| A credential may have been exposed | Revoke the course app in Google Account connections first, delete both Colab Secrets, then reauthorize and store a fresh matching pair. Merely replacing local Secrets does not invalidate a copied old pair. |
| Helper returns 403 | Confirm that you used the exact compatible personal Google Account listed in the private course roster and granted verified email plus at least one of the three displayed health scopes. An old nine-scope credential must be revoked and replaced after the course helper is redeployed. Contact us through an approved private course channel; never post the address, payloads, or credentials. Use the synthetic path while access is resolved. |
| One live data type says its scope was not granted | This is a valid partial-consent result. Single-type calls for granted scopes remain available, but a cell that requests multiple types stops if one required scope is missing. Reauthorize only if you choose to add access, or use the equivalent synthetic data. |
| Authorization stops after consent | The helper cannot issue a credential without its required identity checks and at least one displayed health scope. Use the synthetic path or reauthorize and grant a nonempty subset after reviewing the disclosure. |
| Pagination reports a repeated/invalid token | Stop rather than retrying. Use the synthetic path and report the response status and data type without sharing payloads or credentials. |
| Google Health still returns 401 or 403 | The notebook refreshes once after a 401. If it still fails, stop retrying; reauthorize for 401, or ask us to check scopes and application access for 403. |
| Colab may recycle the runtime | Save a copy in Drive and download the completed notebook before submission. |

Never print or submit a refresh token, helper token, access token, client
secret, or another person's health record.

## TODO: Summary and reflection

1. What did you learn about reading Google Health steps, heart-rate, and sleep
   records and using them in an OpenAI summary? Explain why you chose your date,
   tie one summary claim to an exact displayed value, and name one omission or
   uncertainty.
2. How did you use AI while completing this lab, and what did you verify
   yourself?

Complete the clearly marked answer cell below.

In [ ]:
# TODO: Replace both placeholders with your own run-grounded responses.
topic_reflection = '''
TODO: summarize and reflect on what you learned about this lab's wearable-data
and OpenAI workflow. Include your date choice, one claim tied to an exact
displayed value, and one omission or uncertainty.
'''.strip()
ai_use_reflection = '''
TODO: explain how you used AI and what you verified yourself.
'''.strip()
print(topic_reflection)
print(ai_use_reflection)